# 介紹
本程式使用pyautogui 來完成一個Blitz 的遊戲機器人

In [8]:
import pyautogui as pyg
pyg.sleep(2)
basex=1503
basey=393
gap=40

In [9]:
def gem_position(r, c):
    """回傳第 r 行、第 c 列寶石的螢幕座標 (x, y)；r、c 使用 0 到 7。"""
    x = int(basex + c * gap)
    y = int(basey + r * gap)
    return (x, y)

In [10]:
def get_gem_color(r, c):
    """擷取目前畫面並回傳第 r 行、第 c 列寶石的 RGB 顏色。"""
    im = pyg.screenshot()
    x, y = gem_position(r, c)
    return im.getpixel((x, y))

In [11]:
def can_move_left(r, c, im):
    """判斷 (r, c) 與左側寶石交換後，是否會形成三顆以上連線。"""
    if not (0 <= r < 8 and 0 < c < 8):
        return False

    # 從傳入的截圖取得棋盤各寶石中心顏色。
    colors = [[im.getpixel(gem_position(row, col)) for col in range(8)]
              for row in range(8)]
    colors[r][c], colors[r][c - 1] = colors[r][c - 1], colors[r][c]

    # 檢查交換後兩顆寶石所在位置的橫向、縱向連線。
    for row, col in ((r, c), (r, c - 1)):
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for sign in (-1, 1):
                nr, nc = row + sign * dr, col + sign * dc
                while (0 <= nr < 8 and 0 <= nc < 8
                       and colors[nr][nc] == colors[row][col]):
                    count += 1
                    nr += sign * dr
                    nc += sign * dc
            if count >= 3:
                return True
    return False

In [ ]:
def _can_move_to(r, c, nr, nc, im):
    """模擬與相鄰寶石交換，判斷是否會形成三顆以上連線。"""
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= nr < 8 and 0 <= nc < 8):
        return False

    colors = [[im.getpixel(gem_position(row, col)) for col in range(8)]
              for row in range(8)]
    colors[r][c], colors[nr][nc] = colors[nr][nc], colors[r][c]

    # 只需檢查交換後兩顆寶石所在位置的橫向及縱向連線。
    for row, col in ((r, c), (nr, nc)):
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for sign in (-1, 1):
                check_r, check_c = row + sign * dr, col + sign * dc
                while (0 <= check_r < 8 and 0 <= check_c < 8
                       and colors[check_r][check_c] == colors[row][col]):
                    count += 1
                    check_r += sign * dr
                    check_c += sign * dc
            if count >= 3:
                return True
    return False

def can_move_right(r, c, im):
    """判斷 (r, c) 與右側寶石交換後，是否會形成三顆以上連線。"""
    return _can_move_to(r, c, r, c + 1, im)

def can_move_up(r, c, im):
    """判斷 (r, c) 與上方寶石交換後，是否會形成三顆以上連線。"""
    return _can_move_to(r, c, r - 1, c, im)

def can_move_down(r, c, im):
    """判斷 (r, c) 與下方寶石交換後，是否會形成三顆以上連線。"""
    return _can_move_to(r, c, r + 1, c, im)

In [ ]:
import time

start_time = time.time()
im = pyg.screenshot()
directions = (
    (can_move_left, (0, -1)),
    (can_move_right, (0, 1)),
    (can_move_up, (-1, 0)),
    (can_move_down, (1, 0)),
)

while time.time() - start_time < 60:
    moved = False
    for r in range(8):
        if time.time() - start_time >= 60:
            break
        for c in range(8):
            if time.time() - start_time >= 60:
                break
            for can_move, (dr, dc) in directions:
                if time.time() - start_time >= 60:
                    break
                if can_move(r, c, im):
                    start_x, start_y = gem_position(r, c)
                    end_x, end_y = gem_position(r + dr, c + dc)
                    pyg.moveTo(start_x, start_y)
                    pyg.dragTo(end_x, end_y, duration=0.15, button='left')
                    pyg.sleep(0.3)
                    im = pyg.screenshot()
                    moved = True
                    break
            if moved:
                break
        if moved:
            break
    if not moved:
        pyg.sleep(0.1)
        im = pyg.screenshot()

In [12]:
im=pyg.screenshot()
print (can_move_left(4,6,im))

True


In [13]:
print (gem_position(0,0))
print (gem_position(0,1))
print (gem_position(0,2))

(1503, 393)
(1543, 393)
(1583, 393)


In [14]:
for i in range(8):
    x,y=gem_position(0,i)
    pyg.click(x,y,duration=1)